In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import chi2_contingency

In [11]:
# Load and preprocess the data
selected_avriary_path = "metadata_aviaries/fl_zoo_helsinki_20250624_meta.xlsx"
df = pd.read_excel(selected_avriary_path)

df.drop(columns=["Unnamed: 0.1", "Unnamed: 0", "sessionId", "time", "th1", "th1_value", 'th2', 'th2_value', 'th3', 'th3_value', 'wudate', 'lon', 'lat', 'MIT_AST_label'], inplace=True)
df["fusion_model_prediction"] = df["fusion_model_prediction"].replace("NO PREDICTION (0.0000)", None)
df["Call_Presence"] = df["fusion_model_prediction"].notnull().astype(int)
df

,filename,datetime,precipRate,pressureMax,dewptAvg,windgustHigh,windspeedAvg,tempAve,humidityAvg,winddirAvg,uvHigh,solarRadiationHigh,perch_prediction,birdnet_prediction,overlap,fusion_model_prediction,Call_Presence
0,fl_zoo_helsinki_20250624/0/er_file_2025_06_24_...,2025-06-24 06:12:51,0.0,1018.73,9.7,7.2,3.6,12.7,82.0,287,0,18.1,NaN,NaN,NaN,NaN,0
1,fl_zoo_helsinki_20250624/0/er_file_2025_06_24_...,2025-06-24 06:12:54,0.0,1018.73,9.7,7.2,3.6,12.7,82.0,287,0,18.1,NaN,NaN,NaN,NaN,0
2,fl_zoo_helsinki_20250624/0/er_file_2025_06_24_...,2025-06-24 06:12:56,0.0,1018.73,9.7,7.2,3.6,12.7,82.0,287,0,18.1,NaN,NaN,NaN,NaN,0
3,fl_zoo_helsinki_20250624/0/er_file_2025_06_24_...,2025-06-24 06:12:59,0.0,1018.73,9.7,7.2,3.6,12.7,82.0,287,0,18.1,NaN,NaN,NaN,NaN,0
4,fl_zoo_helsinki_20250624/0/er_file_2025_06_24_...,2025-06-24 06:13:02,0.0,1018.73,9.7,7.2,3.6,12.7,82.0,287,0,18.1,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35128,fl_zoo_helsinki_20250624/351/er_file_2025_07_0...,2025-07-01 07:05:19,0.0,1020.62,14.2,0.0,0.0,19.0,73.2,358,0,37.3,NaN,NaN,NaN,NaN,0
35129,fl_zoo_helsinki_20250624/351/er_file_2025_07_0...,2025-07-01 07:05:21,0.0,1020.62,14.2,0.0,0.0,19.0,73.2,358,0,37.3,NaN,NaN,NaN,NaN,0
35130,fl_zoo_helsinki_20250624/351/er_file_2025_07_0...,2025-07-01 07:05:32,0.0,1020.62,14.2,0.0,0.0,19.0,73.2,358,0,37.3,NaN,NaN,NaN,NaN,0
35131,fl_zoo_helsinki_20250624/351/er_file_2025_07_0...,2025-07-01 07:05:41,0.0,1020.62,14.2,0.0,0.0,19.0,73.2,358,0,37.3,NaN,NaN,NaN,NaN,0


In [ ]:
# See effect of rain on vocalisations
# precipRate we dont know the measure, not explained in the README
rainy_days = df[df["precipRate"] > 0.25] 
rainy_days_calls = rainy_days["fusion_model_prediction"].notnull().sum()

non_rainy_days = df[df["precipRate"] <= 0.25] 
non_rainy_days_calls = non_rainy_days["fusion_model_prediction"].notnull().sum()

print(f'Rainy days vocalisations: {rainy_days_calls/len(rainy_days)*100:.2f}%')
print(f'Non-rainy days vocalisations: {non_rainy_days_calls/len(non_rainy_days)*100:.2f}%')


df['rain'] = (df['precipRate'] > 0.25).astype(int)
contingency_table = pd.crosstab(df['rain'], df['Call_Presence'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(contingency_table)
print(f"Chi-squared: {chi2:.4f}, p-value: {p_value:.4f}") # Not sure this test is correct

Rainy days vocalisations: 35.69%
Non-rainy days vocalisations: 27.86%
Call_Presence      0     1
rain                      
0              24693  9535
1                582   323
Chi-squared: 26.4146, p-value: 0.0000


In [28]:
# Look at the impact of temperature on vocalisations
print(f"Maximum temperature: {df['tempAve'].max()}, Minimum temperature: {df['tempAve'].min()}\n")

bins = [df['tempAve'].min(), 20, df['tempAve'].max()]
bins = [i for i in range(int(df['tempAve'].min()), int(df['tempAve'].max()) + 1, 5)]

df['temp_category'] = pd.cut(df['tempAve'], bins=bins)
counts = df.groupby('temp_category')['Call_Presence'].count()/len(df)

print(counts)

Maximum temperature: 34.6, Minimum temperature: 12.7

temp_category
(12, 17]    0.083853
(17, 22]    0.369197
(22, 27]    0.249822
(27, 32]    0.199613
Name: Call_Presence, dtype: float64


/tmp/ipykernel_4559/2643728871.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = df.groupby('temp_category')['Call_Presence'].count()/len(df)
